# Lab: Random Forests and XGBoost

*This notebook demonstrates a head-to-head comparison of two powerful ensemble models, Random Forest and XGBoost, on a regression task, guided by the [Foundational Methodology for Data Science](../../../05_methodology/). While real-world projects require comprehensive documentation at each stage, this lab focuses on a streamlined, practical approach to demonstrate the core concepts and workflow. Therefore we will settle with the summaries of hypothetical stage reports.*

---

> ### 📝 1. Business Understanding Report (Summary)
>
> - **Business Context:** This lab addresses the practical need for more accurate and robust predictive analytics in real estate, using California housing data as a proxy. Real-world decision-makers—such as real estate investors, property managers, or urban planners—require precise and reliable models to estimate home values and understand price drivers across diverse neighborhoods and market conditions.
> - **Business Problem:** Traditional models (like linear regression or even a single regression tree) can struggle with nonlinearity, outliers, and the complex interplay between features, leading to either underfitting or overfitting, and resulting in poor generalization on new data. Stakeholders need a solution that can leverage complex patterns without succumbing to overfitting or high variance.
> - **Project Goal:** To benchmark and compare the predictive performance and interpretability of two popular ensemble modeling approaches—**Random Forests** and **XGBoost**—in estimating California district-level median home values. The lab will use feature-rich, block-group-level data to demonstrate how ensemble models can improve accuracy, reduce model error, and provide actionable insights for business stakeholders (e.g., identifying key determinants of housing prices).
> - **Key Questions:**  
>    - How do Random Forests and XGBoost differ in terms of accuracy, bias-variance tradeoff, feature importance, and prediction stability on the same regression problem?
>    - Can these advanced ensemble techniques deliver practical, explainable value that justifies their complexity over traditional models?
> - **Success Criteria:** Success will be evaluated by considering out-of-sample prediction error (R² and RMSE), the ability to generalize to unseen data, clarity of feature importance rankings, and the ease with which results can be communicated to non-technical stakeholders.
>

---

> ### 📝 2. Analytic Approach Report (Summary)
>
> - **Problem Type:**  Supervised machine learning — specifically, *regression*, as the goal is to predict a continuous target variable (median house value) using multiple numeric and spatial features.
> - **Model Selection:** This lab will benchmark and compare two leading ensemble regression algorithms:
>   - **Random Forest Regressor** (`RandomForestRegressor` from scikit-learn): An ensemble of decision trees trained using bootstrap aggregation (bagging) and random feature selection, designed to reduce variance and improve generalization.
>   - **XGBoost Regressor** (`XGBRegressor` from xgboost package): An advanced gradient boosting method that iteratively builds additive decision trees, focusing on correcting the previous trees’ errors, and is known for its high accuracy and efficiency.
> - **Evaluation Metrics:**  
>    - **R-squared (R²)**: Measures the proportion of variance in housing prices explained by the model.  
>    - **Root Mean Squared Error (RMSE)**: Quantifies average prediction error in the original price units, for direct business interpretation.  
>    - **Feature Importance**: Both ensemble models supply importance scores for each feature, helping stakeholders interpret which variables drive home values.
> - **Motivation for Model Choice:** Ensemble tree-based models are chosen for their ability to model nonlinear relationships, handle outliers, and automatically capture complex feature interactions. Comparing bagging (Random Forest) to boosting (XGBoost) provides insight into bias-variance tradeoff management in modern ML workflows. The head-to-head setup also illustrates the relative value and interpretability of each model class for a typical real-world tabular regression problem.
> - **Practical Considerations:** In practice, hyperparameter tuning (e.g., number of trees, maximum depth) and rigorous cross-validation should be performed to optimize model performance and reliability. The experimental workflow will include data preprocessing (if necessary), model training on train data, and careful evaluation on held-out test data to accurately assess generalization.

---

> ### 📝 3. Data Requirements Report (Summary)
>
> - **Data Source:** The lab will use the **California Housing dataset** from scikit-learn, a widely-used, fully anonymized, and clean dataset. It aggregates socioeconomic and geographic features for over 20,000 California census block groups, and is ideal for regression model demonstration and benchmarking.
> - **Features (Independent Variables):**  
>    - **med_income:** Median household income in each district  
>    - **med_age:** Median age of houses  
>    - **avg_rooms:** Average number of rooms per household  
>    - **avg_bedrooms:** Average number of bedrooms per household  
>    - **population:** Block group population  
>    - **avg_occup:** Average occupancy  
>    - **lat:** Latitude  
>    - **long:** Longitude  
> - **Target (Dependent Variable):**  
>    - **med_value:** Median house value in each California district (in $100,000s)
> - **Data Granularity & Privacy:** Each observation summarizes statistics for a census-defined block group, not individuals. Data is fully aggregated and contains no personally identifiable information.
> - **Preprocessing Considerations:**  No missing values are present in the raw dataset, and features are already numeric and suitable for tree-based models. Standardization or normalization is not strictly required for Random Forests or XGBoost, but exploratory statistics and outlier checks will be performed as part of good modeling practice.
> - **Data Partitioning:**  Data will be split into **training** and **test** sets to allow for unbiased generalization assessment. (The split ratio will be specified during model development.)

---

## Stage 4: Data Collection
The data was already extracted, transformed, and loaded during the [Regression Trees Lab](./13_regression_trees_lab.ipynb). Therefore we'll just import the necessary libraries and load our interim data to a new DataFrame. 

In [4]:
# Necessary imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurations
plt.style.use("fivethirtyeight")
sns.set_theme(style="white", palette="colorblind")

In [5]:
# Define and create the interim data directory
interim_data_dir = Path("../data/interim")
interim_data_path = interim_data_dir / "california_housing_interim_v1.parquet"

# Load the final interim data and verify its contents
try:
    df_verified = pd.read_parquet(interim_data_path)
    print(f"Verification successful. The following DataFrame is ready for analysis with {df_verified.shape[0]} samples and {df_verified.shape[1]} features:")
    display(df_verified.sample(5))
except Exception as e:
    print(f"An error occurred while verifying the interim data: {e}")

Verification successful. The following DataFrame is ready for analysis with 20640 samples and 9 features:


,med_income,med_age,avg_rooms,avg_bedrooms,population,avg_occup,lat,long,med_value
11977,4.4063,15.0,6.104430,1.003165,1237.0,3.914557,34.00,-117.50,1.285
19535,3.0577,13.0,6.056086,1.167064,3033.0,3.619332,37.65,-120.94,1.190
14283,0.6433,24.0,3.725664,0.893805,396.0,3.504425,32.71,-117.12,1.113
11263,3.9566,35.0,5.068282,1.028634,2272.0,5.004405,33.79,-117.99,1.678
12349,3.6667,15.0,14.304762,2.928571,437.0,2.080952,33.80,-116.48,0.900



---

> ### 📝 4. Data Collection Report (Summary)
>
> - **Extraction:**  The required dataset was already extracted and saved during the Regression Trees Lab, using the official California Housing data from scikit-learn.
> - **Transformation:**  All preprocessing—column renaming, data quality checks, and saving in analysis-ready `parquet` format—was previously completed. No additional data cleaning or preparation is performed in this stage to maintain consistency across modeling experiments.
> - **Loading:**  In this lab, the interim (preprocessed) dataset is loaded directly from the saved file (`/data/interim/california_housing_interim_v1.parquet`) into a new DataFrame, ensuring that all ensembles and baseline models train on exactly the same data as previous labs.
> - **Verification:**  The integrity, shape, and sample contents of the loaded interim DataFrame are checked and displayed before proceeding with any modeling or analysis, guaranteeing that the data available at this stage matches prior preparation steps.

---

> ### 📝 5. Data Understanding Report (Summary)
>
> _Note: A full exploratory data analysis (EDA) including checks for feature distributions, outliers, and variable relationships was performed in the Regression Trees Lab. The following is a summary of key findings relevant to ensemble modeling._
>
> - **Feature Distributions & Outliers:** Most features—including median income, average rooms, bedrooms, and occupancy—are strongly right-skewed with outliers and heavy tails at high values. The target variable (`med_value`) is also right-skewed, with a visible spike at its upper limit. Regression trees are robust to these issues; no transformations or outlier removal are necessary.
> - **Relationships with the Target:**  
>   - **Median Income (`med_income`)** is the clearest and strongest predictor, with home values increasing sharply above a threshold (~$5,000). This makes it a prime candidate for early tree splits.
>   - **Average Rooms/Bedrooms** show positive association with value, but with flattening at extremes, indicating possible mid-range split points.
>   - **Average Occupancy** features a clear threshold effect: extremely high occupancy is linked to much lower home values.
>   - **Population** shows little direct association except at the highest values, where prices may dip.
>   - **Geography (`lat`, `long`)**: Despite having weak linear correlations with the target, spatial plots reveal striking regional clusters in home values—suggesting regression trees can take advantage of location-based splits. Higher house prices concentrate along the coast and in urban centers.
> - **Feature Redundancy:** Some variables (e.g., average rooms and bedrooms) are highly correlated. However, this is not problematic for regression trees, which naturally select among correlated features.
> - **Summary for Modeling:** The dataset structure and relationships observed strongly support the use of regression trees:
>   - Trees are well-suited for capturing sharp value jumps at thresholds and spatial boundaries.
>   - No pre-processing is needed for outliers, skew, or feature scaling.
>   - The most informative variables for splitting are likely to be median income, average rooms, occupancy, and geographic coordinates.
